# Residual Momentum

## Research question

Does stock-specific momentum remain predictive after removing the component of returns explained by the broad equity market?

For each stock, daily returns are decomposed using a rolling market model:

$$
r_{i,t}
=
\alpha_{i,t-1}
+
\beta_{i,t-1}r_{m,t}
+
\varepsilon_{i,t}.
$$

The regression coefficients used for day \(t\) are estimated using information available only through day \(t-1\).

The primary factor is the sum of residual returns over the conventional 12–1 momentum formation period:

$$
\text{Residual Momentum}_{i,t}
=
\sum_{s=t-251}^{t-21}\varepsilon_{i,s}.
$$

The most recent 21 trading days are excluded. The primary test is whether residual momentum has predictive information beyond conventional 12–1 momentum.

## 1. Setup and data audit

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

current = Path.cwd().resolve()
for _ in range(10):
    if (current / "pyproject.toml").exists():
        ROOT = current
        break
    current = current.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

factor_panel = pd.read_parquet(
    ROOT / "data" / "processed" / "factor_panel.parquet"
)

required_columns = [
    "date",
    "ticker",
    "adj_close",
    "mom_12_1m_raw",
]

missing_columns = [
    column
    for column in required_columns
    if column not in factor_panel.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

residual_momentum_panel = (
    factor_panel
    .copy()
    .sort_values(["ticker", "date"])
)

residual_momentum_panel["date"] = pd.to_datetime(
    residual_momentum_panel["date"]
)

assert not residual_momentum_panel.duplicated(
    ["date", "ticker"]
).any()

In [2]:
residual_momentum_panel["stock_return_1d"] = (
    residual_momentum_panel
    .groupby("ticker")["adj_close"]
    .pct_change(fill_method=None)
)

In [3]:
spy_path = ROOT / "data" / "raw" / "spy_benchmark.parquet"

assert spy_path.exists(), f"SPY benchmark not found: {spy_path}"

spy_data = pd.read_parquet(spy_path).copy()

spy_data["date"] = pd.to_datetime(spy_data["date"]).dt.tz_localize(None)

spy_data = spy_data.sort_values("date").drop_duplicates("date", keep="last")

spy_returns = spy_data[["date", "adj_close"]].rename(columns={"adj_close": "spy_close"})

# Historical return ending on date t.
spy_returns["spy_return_1d"] = spy_returns["spy_close"].pct_change(fill_method=None)

residual_momentum_panel = residual_momentum_panel.merge(
    spy_returns[["date", "spy_return_1d"]],
    on="date",
    how="left",
    validate="many_to_one",
)

In [4]:
return_alignment_summary = pd.Series(
    {
        "panel_rows": len(residual_momentum_panel),
        "stock_return_coverage": (
            residual_momentum_panel["stock_return_1d"]
            .notna()
            .mean()
        ),
        "spy_return_coverage": (
            residual_momentum_panel["spy_return_1d"]
            .notna()
            .mean()
        ),
        "dates_missing_spy_return": (
            residual_momentum_panel
            .loc[
                residual_momentum_panel["spy_return_1d"].isna(),
                "date",
            ]
            .nunique()
        ),
    },
    name="value",
)

return_alignment_summary

panel_rows                  284249.000000
stock_return_coverage            0.999645
spy_return_coverage              0.999659
dates_missing_spy_return         1.000000
Name: value, dtype: float64

## 2. Signal construction

### 2.1 Lagged rolling market models

In [5]:
BETA_WINDOW = 126
BETA_MIN_PERIODS = 100


def add_lagged_market_model(group):
    group = group.sort_values("date").copy()

    stock_returns = group["stock_return_1d"]
    market_returns = group["spy_return_1d"]

    rolling_market_variance = market_returns.rolling(
        BETA_WINDOW,
        min_periods=BETA_MIN_PERIODS,
    ).var()

    rolling_covariance = stock_returns.rolling(
        BETA_WINDOW,
        min_periods=BETA_MIN_PERIODS,
    ).cov(market_returns)

    rolling_beta = rolling_covariance / rolling_market_variance

    rolling_alpha = (
        stock_returns.rolling(
            BETA_WINDOW,
            min_periods=BETA_MIN_PERIODS,
        ).mean()
        - rolling_beta
        * market_returns.rolling(
            BETA_WINDOW,
            min_periods=BETA_MIN_PERIODS,
        ).mean()
    )

    # The return on date t uses coefficients estimated through t-1.
    group["market_model_alpha_lag1"] = rolling_alpha.shift(1)
    group["market_model_beta_lag1"] = rolling_beta.shift(1)

    group["market_model_residual"] = group["stock_return_1d"] - (
        group["market_model_alpha_lag1"]
        + group["market_model_beta_lag1"] * group["spy_return_1d"]
    )

    return group


residual_momentum_panel = pd.concat(
    [
        add_lagged_market_model(group)
        for _, group in residual_momentum_panel.groupby(
            "ticker",
            sort=False,
        )
    ],
    ignore_index=True,
).sort_values(["ticker", "date"])

### 2.2 Construct 12–1 residual momentum

In [6]:
FORMATION_WINDOW = 231
SKIP_WINDOW = 21
FORMATION_MIN_PERIODS = 200

residual_momentum_panel[
    "residual_momentum_12_1_raw"
] = (
    residual_momentum_panel
    .groupby("ticker")["market_model_residual"]
    .transform(
        lambda series: (
            series
            .shift(SKIP_WINDOW)
            .rolling(
                FORMATION_WINDOW,
                min_periods=FORMATION_MIN_PERIODS,
            )
            .sum()
        )
    )
)

## 3. Construction diagnostics

In [7]:
residual_momentum_coverage = pd.Series(
    {
        "total_rows": len(residual_momentum_panel),
        "residual_return_count": (
            residual_momentum_panel["market_model_residual"].notna().sum()
        ),
        "residual_return_coverage": (
            residual_momentum_panel["market_model_residual"].notna().mean()
        ),
        "residual_momentum_count": (
            residual_momentum_panel["residual_momentum_12_1_raw"].notna().sum()
        ),
        "residual_momentum_coverage": (
            residual_momentum_panel["residual_momentum_12_1_raw"].notna().mean()
        ),
        "tickers_with_signal": (
            residual_momentum_panel.loc[
                residual_momentum_panel["residual_momentum_12_1_raw"].notna(),
                "ticker",
            ].nunique()
        ),
        "dates_with_signal": (
            residual_momentum_panel.loc[
                residual_momentum_panel["residual_momentum_12_1_raw"].notna(),
                "date",
            ].nunique()
        ),
    },
    name="value",
)

residual_momentum_coverage

total_rows                    284249.000000
residual_return_count         274136.000000
residual_return_coverage           0.964422
residual_momentum_count       252136.000000
residual_momentum_coverage         0.887025
tickers_with_signal              100.000000
dates_with_signal               2570.000000
Name: value, dtype: float64

In [8]:
residual_momentum_distribution = (
    residual_momentum_panel[
        [
            "market_model_alpha_lag1",
            "market_model_beta_lag1",
            "market_model_residual",
            "residual_momentum_12_1_raw",
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
    .T
)

residual_momentum_distribution

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
market_model_alpha_lag1,274136.0,0.000182,0.001364,-0.007881,-0.003242,-0.001847,-0.000590,0.000157,0.000881,0.002293,0.004186,0.012578
market_model_beta_lag1,274136.0,0.966148,0.489109,-0.817177,-0.109624,0.194853,0.647942,0.963076,1.239971,1.796200,2.419549,3.794389
market_model_residual,274136.0,-0.000019,0.015761,-0.345596,-0.043307,-0.022000,-0.006934,-0.000149,0.006704,0.022210,0.044523,0.617262
residual_momentum_12_1_raw,252136.0,-0.004312,0.140106,-0.942733,-0.347836,-0.215291,-0.085916,-0.005573,0.071351,0.215127,0.396016,1.120626


### 3.1 Similarity to conventional momentum

In [9]:
daily_momentum_correlations = (
    residual_momentum_panel
    .dropna(
        subset=[
            "residual_momentum_12_1_raw",
            "mom_12_1m_raw",
        ]
    )
    .groupby("date")
    .apply(
        lambda group: group[
            [
                "residual_momentum_12_1_raw",
                "mom_12_1m_raw",
            ]
        ].corr(method="spearman").iloc[0, 1],
    )
    .rename("spearman_correlation")
)

momentum_correlation_summary = (
    daily_momentum_correlations
    .describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    )
)

momentum_correlation_summary

count    2570.000000
mean        0.423562
std         0.177785
min        -0.136982
5%          0.147128
25%         0.297214
50%         0.424519
75%         0.553074
95%         0.723152
max         0.804650
Name: spearman_correlation, dtype: float64

### 3.2 Residual correlation with SPY

In [10]:
residual_market_correlations = (
    residual_momentum_panel
    .dropna(
        subset=[
            "market_model_residual",
            "spy_return_1d",
        ]
    )
    .groupby("ticker")
    .apply(
        lambda group: group[
            [
                "market_model_residual",
                "spy_return_1d",
            ]
        ].corr().iloc[0, 1],
    )
    .rename("residual_market_correlation")
)

residual_market_correlation_summary = (
    residual_market_correlations.describe()
)

residual_market_correlation_summary

count    100.000000
mean       0.003068
std        0.025469
min       -0.097540
25%       -0.010852
50%        0.002676
75%        0.018846
max        0.071615
Name: residual_market_correlation, dtype: float64

## 4. Predictive testing

### 4.1 Prepare signal variants

In [29]:
required_columns = [
    "date",
    "ticker",
    "sector",
    "residual_momentum_12_1_raw",
    "mom_12_1m_raw",
    "forward_ret_1d",
    "forward_ret_5d",
]

missing_columns = [
    column
    for column in required_columns
    if column not in residual_momentum_panel.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

residual_momentum_test_panel = (
    residual_momentum_panel
    .copy()
    .sort_values(["date", "ticker"])
)

In [30]:
def cross_sectional_winsorise(
    series,
    lower_quantile=0.01,
    upper_quantile=0.99,
):
    valid = series.dropna()

    if len(valid) < 10:
        return pd.Series(np.nan, index=series.index)

    lower = valid.quantile(lower_quantile)
    upper = valid.quantile(upper_quantile)

    return series.clip(lower=lower, upper=upper)


def cross_sectional_zscore(series):
    mean = series.mean()
    std = series.std()

    if pd.isna(std) or std == 0:
        return pd.Series(np.nan, index=series.index)

    return (series - mean) / std

In [31]:
residual_momentum_test_panel[
    "residual_momentum_12_1_winsorised"
] = (
    residual_momentum_test_panel
    .groupby("date")["residual_momentum_12_1_raw"]
    .transform(cross_sectional_winsorise)
)

residual_momentum_test_panel[
    "residual_momentum_12_1_z"
] = (
    residual_momentum_test_panel
    .groupby("date")[
        "residual_momentum_12_1_winsorised"
    ]
    .transform(cross_sectional_zscore)
)

residual_momentum_test_panel[
    "residual_momentum_12_1_sector_demeaned"
] = (
    residual_momentum_test_panel[
        "residual_momentum_12_1_winsorised"
    ]
    -
    residual_momentum_test_panel
    .groupby(["date", "sector"])[
        "residual_momentum_12_1_winsorised"
    ]
    .transform("mean")
)

residual_momentum_test_panel[
    "residual_momentum_12_1_sector_neutral_z"
] = (
    residual_momentum_test_panel
    .groupby("date")[
        "residual_momentum_12_1_sector_demeaned"
    ]
    .transform(cross_sectional_zscore)
)

In [32]:
residual_momentum_test_panel[
    "mom_12_1m_winsorised"
] = (
    residual_momentum_test_panel
    .groupby("date")["mom_12_1m_raw"]
    .transform(cross_sectional_winsorise)
)

residual_momentum_test_panel["mom_12_1m_z"] = (
    residual_momentum_test_panel
    .groupby("date")["mom_12_1m_winsorised"]
    .transform(cross_sectional_zscore)
)

### 4.2 IC utilities

In [33]:
def calculate_daily_ic(
    panel,
    signal_column,
    return_column,
    minimum_stocks=20,
):
    def calculate_one_date(group):
        valid = group[
            [signal_column, return_column]
        ].dropna()

        if len(valid) < minimum_stocks:
            return np.nan

        if (
            valid[signal_column].nunique() < 2
            or valid[return_column].nunique() < 2
        ):
            return np.nan

        return valid[signal_column].corr(
            valid[return_column],
            method="spearman",
        )

    return (
        panel
        .groupby("date")
        .apply(
            calculate_one_date,
            include_groups=False,
        )
        .dropna()
        .rename("ic")
    )


def summarise_ic(ic_series):
    ic_series = ic_series.dropna()
    count = len(ic_series)
    std = ic_series.std()

    return pd.Series(
        {
            "count": count,
            "mean_ic": ic_series.mean(),
            "std_ic": std,
            "ic_ir": (
                ic_series.mean() / std
                if std > 0
                else np.nan
            ),
            "t_stat": (
                ic_series.mean()
                / std
                * np.sqrt(count)
                if std > 0 and count > 0
                else np.nan
            ),
            "positive_fraction": (
                (ic_series > 0).mean()
                if count > 0
                else np.nan
            ),
        }
    )

### 4.3 Horizon IC: raw and sector-neutral

In [34]:
horizon_test_specifications = {
    "Raw — 1-day": (
        "residual_momentum_12_1_z",
        "forward_ret_1d",
    ),
    "Raw — 5-day": (
        "residual_momentum_12_1_z",
        "forward_ret_5d",
    ),
    "Sector neutral — 1-day": (
        "residual_momentum_12_1_sector_neutral_z",
        "forward_ret_1d",
    ),
    "Sector neutral — 5-day": (
        "residual_momentum_12_1_sector_neutral_z",
        "forward_ret_5d",
    ),
}

residual_momentum_ic_series = {}
horizon_ic_rows = []

for test_name, (
    signal_column,
    return_column,
) in horizon_test_specifications.items():

    daily_ic = calculate_daily_ic(
        residual_momentum_test_panel,
        signal_column,
        return_column,
    )

    residual_momentum_ic_series[test_name] = daily_ic

    summary = summarise_ic(daily_ic)
    summary.name = test_name
    horizon_ic_rows.append(summary)

residual_momentum_horizon_ic_summary = pd.DataFrame(
    horizon_ic_rows
)

residual_momentum_horizon_ic_summary

,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
Raw — 1-day,2569.0,0.009459,0.215658,0.043860,2.223071,0.519657
Raw — 5-day,2565.0,0.005611,0.205977,0.027239,1.379523,0.522807
Sector neutral — 1-day,2569.0,0.008187,0.155827,0.052537,2.662878,0.522382
Sector neutral — 5-day,2565.0,0.007520,0.149377,0.050345,2.549788,0.526316


### 4.4 Subperiod stability

In [35]:
subperiods = {
    "2015–2018": ("2015-01-01", "2018-12-31"),
    "2019–2022": ("2019-01-01", "2022-12-31"),
    "2023–present": ("2023-01-01", None),
}

subperiod_ic_rows = []

for signal_name, signal_column in {
    "Raw": "residual_momentum_12_1_z",
    "Sector neutral": (
        "residual_momentum_12_1_sector_neutral_z"
    ),
}.items():

    for period_name, (
        start_date,
        end_date,
    ) in subperiods.items():

        period_panel = residual_momentum_test_panel.loc[
            residual_momentum_test_panel["date"]
            >= pd.Timestamp(start_date)
        ]

        if end_date is not None:
            period_panel = period_panel.loc[
                period_panel["date"]
                <= pd.Timestamp(end_date)
            ]

        period_ic = calculate_daily_ic(
            period_panel,
            signal_column,
            "forward_ret_5d",
        )

        summary = summarise_ic(period_ic).to_dict()
        summary["signal"] = signal_name
        summary["subperiod"] = period_name

        subperiod_ic_rows.append(summary)

residual_momentum_subperiod_ic_summary = (
    pd.DataFrame(subperiod_ic_rows)
    .set_index(["signal", "subperiod"])
)

residual_momentum_subperiod_ic_summary

count   mean_ic    std_ic     ic_ir    t_stat  \
signal         subperiod                                                      
Raw            2015–2018      685.0  0.002084  0.196483  0.010606  0.277576   
               2019–2022     1008.0  0.000427  0.218526  0.001952  0.061977   
               2023–present   872.0  0.014373  0.198110  0.072552  2.142448   
Sector neutral 2015–2018      685.0  0.016378  0.138584  0.118184  3.093163   
               2019–2022     1008.0 -0.006524  0.153171 -0.042591 -1.352215   
               2023–present   872.0  0.016797  0.151994  0.110508  3.263277   

                             positive_fraction  
signal         subperiod                        
Raw            2015–2018              0.497810  
               2019–2022              0.513889  
               2023–present           0.552752  
Sector neutral 2015–2018              0.543066  
               2019–2022              0.489087  
               2023–present           0.556193

### 4.5 Non-overlapping five-day IC

In [36]:
available_dates = np.array(
    sorted(
        residual_momentum_test_panel.loc[
            residual_momentum_test_panel[
                "residual_momentum_12_1_z"
            ].notna(),
            "date",
        ].unique()
    )
)

non_overlapping_ic_rows = []

for signal_name, signal_column in {
    "Raw": "residual_momentum_12_1_z",
    "Sector neutral": (
        "residual_momentum_12_1_sector_neutral_z"
    ),
}.items():

    for offset in range(5):
        selected_dates = available_dates[offset::5]

        offset_panel = residual_momentum_test_panel[
            residual_momentum_test_panel["date"].isin(
                selected_dates
            )
        ]

        offset_ic = calculate_daily_ic(
            offset_panel,
            signal_column,
            "forward_ret_5d",
        )

        summary = summarise_ic(offset_ic).to_dict()
        summary["signal"] = signal_name
        summary["offset"] = offset

        non_overlapping_ic_rows.append(summary)

residual_momentum_non_overlapping_ic = (
    pd.DataFrame(non_overlapping_ic_rows)
    .set_index(["signal", "offset"])
)

residual_momentum_non_overlapping_ic

count   mean_ic    std_ic     ic_ir    t_stat  \
signal         offset                                                  
Raw            0       513.0  0.008821  0.207949  0.042419  0.960760   
               1       513.0  0.007426  0.211717  0.035073  0.794381   
               2       513.0  0.003126  0.202314  0.015451  0.349955   
               3       513.0  0.000456  0.203405  0.002241  0.050766   
               4       513.0  0.008224  0.205033  0.040112  0.908524   
Sector neutral 0       513.0  0.011012  0.150370  0.073235  1.658739   
               1       513.0  0.007189  0.154620  0.046494  1.053070   
               2       513.0  0.004389  0.148019  0.029654  0.671639   
               3       513.0  0.005407  0.145971  0.037042  0.838972   
               4       513.0  0.009605  0.148239  0.064792  1.467496   

                       positive_fraction  
signal         offset                     
Raw            0                0.534113  
               1                0.522417  
               2                0.528265  
               3                0.512671  
               4                0.516569  
Sector neutral 0                0.532164  
               1                0.512671  
               2                0.500975  
               3                0.534113  
               4                0.551657

In [37]:
non_overlapping_consistency_summary = (
    residual_momentum_non_overlapping_ic
    .reset_index()
    .groupby("signal")
    .agg(
        mean_ic_across_offsets=("mean_ic", "mean"),
        min_mean_ic=("mean_ic", "min"),
        max_mean_ic=("mean_ic", "max"),
        mean_t_stat=("t_stat", "mean"),
        positive_offsets=(
            "mean_ic",
            lambda values: int((values > 0).sum()),
        ),
    )
)

non_overlapping_consistency_summary

,mean_ic_across_offsets,min_mean_ic,max_mean_ic,mean_t_stat,positive_offsets
signal,,,,,
Raw,0.005611,0.000456,0.008821,0.612877,5
Sector neutral,0.007520,0.004389,0.011012,1.137983,5


### 4.6 Incremental information beyond momentum

In [38]:
residual_momentum_test_panel = (
    residual_momentum_test_panel
    .reset_index(drop=True)
    .copy()
)

residual_momentum_test_panel[
    "residual_momentum_incremental"
] = np.nan

for _, group_indices in (
    residual_momentum_test_panel
    .groupby("date", sort=False)
    .groups
    .items()
):
    valid_indices = (
        residual_momentum_test_panel
        .loc[
            group_indices,
            [
                "residual_momentum_12_1_z",
                "mom_12_1m_z",
            ],
        ]
        .dropna()
        .index
    )

    if len(valid_indices) < 20:
        continue

    dependent = residual_momentum_test_panel.loc[
        valid_indices,
        "residual_momentum_12_1_z",
    ].to_numpy()

    raw_momentum = residual_momentum_test_panel.loc[
        valid_indices,
        "mom_12_1m_z",
    ].to_numpy()

    if np.isclose(np.std(raw_momentum), 0):
        continue

    design_matrix = np.column_stack(
        [
            np.ones(len(raw_momentum)),
            raw_momentum,
        ]
    )

    coefficients = np.linalg.lstsq(
        design_matrix,
        dependent,
        rcond=None,
    )[0]

    incremental_signal = (
        dependent
        - design_matrix @ coefficients
    )

    residual_momentum_test_panel.loc[
        valid_indices,
        "residual_momentum_incremental",
    ] = incremental_signal

In [39]:
residual_momentum_test_panel[
    "residual_momentum_incremental_z"
] = (
    residual_momentum_test_panel
    .groupby("date")[
        "residual_momentum_incremental"
    ]
    .transform(cross_sectional_zscore)
)

In [40]:
incremental_orthogonality = (
    residual_momentum_test_panel
    .dropna(
        subset=[
            "residual_momentum_incremental_z",
            "mom_12_1m_z",
        ]
    )
    .groupby("date")
    .apply(
        lambda group: group[
            [
                "residual_momentum_incremental_z",
                "mom_12_1m_z",
            ]
        ].corr(method="pearson").iloc[0, 1],
    )
    .describe()
)

incremental_orthogonality

count    2.570000e+03
mean    -1.717814e-17
std      1.851695e-16
min     -7.702172e-16
25%     -1.100886e-16
50%     -9.156479e-18
75%      7.339021e-17
max      1.121552e-15
dtype: float64

In [41]:
common_sample = residual_momentum_test_panel.dropna(
    subset=[
        "mom_12_1m_z",
        "residual_momentum_12_1_z",
        "residual_momentum_incremental_z",
        "forward_ret_1d",
        "forward_ret_5d",
    ]
)

incremental_test_rows = []

for signal_name, signal_column in {
    "Raw 12–1 momentum": "mom_12_1m_z",
    "Residual momentum": "residual_momentum_12_1_z",
    "Residual momentum incremental": (
        "residual_momentum_incremental_z"
    ),
}.items():

    for horizon_name, return_column in {
        "1-day": "forward_ret_1d",
        "5-day": "forward_ret_5d",
    }.items():

        daily_ic = calculate_daily_ic(
            common_sample,
            signal_column,
            return_column,
        )

        summary = summarise_ic(daily_ic).to_dict()
        summary["signal"] = signal_name
        summary["horizon"] = horizon_name

        incremental_test_rows.append(summary)

residual_momentum_incremental_ic_summary = (
    pd.DataFrame(incremental_test_rows)
    .set_index(["signal", "horizon"])
)

residual_momentum_incremental_ic_summary

count   mean_ic    std_ic     ic_ir  \
signal                        horizon                                         
Raw 12–1 momentum             1-day    2565.0  0.020241  0.283618  0.071368   
                              5-day    2565.0  0.021385  0.277385  0.077096   
Residual momentum             1-day    2565.0  0.009613  0.215661  0.044576   
                              5-day    2565.0  0.005611  0.205977  0.027239   
Residual momentum incremental 1-day    2565.0 -0.000058  0.190248 -0.000307   
                              5-day    2565.0 -0.007079  0.187010 -0.037853   

                                         t_stat  positive_fraction  
signal                        horizon                               
Raw 12–1 momentum             1-day    3.614494           0.541520  
                              5-day    3.904577           0.559844  
Residual momentum             1-day    2.257584           0.519688  
                              5-day    1.379523           0.522807  
Residual momentum incremental 1-day   -0.015560           0.494737  
                              5-day   -1.917079           0.473684

## 5. Conclusion

Residual momentum passes the construction checks but does not qualify for promotion as a core factor.

The rolling market model successfully removes broad market co-movement: residual returns have approximately zero correlation with SPY. The resulting signal also has good coverage and is meaningfully distinct from conventional 12–1 momentum, with an average cross-sectional Spearman correlation of approximately 0.42.

However, its predictive performance is weak:

- Raw 5-day IC is 0.0056 ($t=1.38$).
- Sector-neutral 5-day IC improves to 0.0075 ($t=2.55$).
- All five non-overlapping offsets are positive, but their individual evidence remains modest.
- Performance is unstable across subperiods, including a negative sector-neutral IC during 2019–2022.
- After removing its cross-sectional relationship with conventional momentum, the incremental component has a 5-day IC of -0.0071 ($t=-1.92$).

The incremental test is decisive. Although residual momentum is different from conventional momentum, its distinct component does not provide positive complementary information at the target horizon.

**Decision:** do not promote residual momentum or proceed to portfolio-level backtesting. Retain conventional 12–1 momentum as the core momentum factor.

The negative incremental IC raises a possible follow-up hypothesis: the inverted incremental signal could act as a penalty term within a momentum composite,

$$
S_{i,t}
=
MOM_{i,t}
-
\lambda RM^\perp_{i,t}.
$$

This interpretation is only exploratory. The current evidence is marginal and was identified after inspecting the results, so testing it immediately would introduce additional data-mining risk. Record the idea for a later, explicitly out-of-sample robustness study after the fundamental research workflow has been completed.

### 5.1 Research lesson

Signal distinctiveness is necessary but not sufficient. A factor can differ substantially from an established signal while still offering no useful incremental predictive information. Incremental testing should therefore remain an early promotion gate, before portfolio construction and transaction-cost analysis.